# 04 Retrain DwellObserver-T

Runs a quick smoke DwellObserver-T training by default. With `RUN_MODE=paper`, repeats the corrected three-seed EMA teacher configuration.

In [2]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_DIR = Path(os.environ.get("AMPERE_DATA_DIR", "data/raw"))
RUN_MODE = os.environ.get("AMPERE_RUN_MODE", "smoke")
SEEDS = [42, 123, 2026]
OUTPUT_DIR = Path(os.environ.get("AMPERE_OUTPUT_DIR", "runs"))

try:
    display
except NameError:
    display = print

print("ROOT= <repo root>")
print("DATA_DIR=", DATA_DIR)
print("RUN_MODE=", RUN_MODE)
print("OUTPUT_DIR=", OUTPUT_DIR)

ROOT= <repo root>
DATA_DIR= data\raw
RUN_MODE= smoke
OUTPUT_DIR= runs


In [3]:
from ampere_public.publication import build_public_canonical_outputs, run_command
import sys

build_public_canonical_outputs(DATA_DIR, OUTPUT_DIR / "processed")
seeds = SEEDS if RUN_MODE == "paper" else [42]
for seed in seeds:
    out = OUTPUT_DIR / "reconstruction" / f"dwellobserver_t_seed{seed}"
    fig = OUTPUT_DIR / "figures" / f"dwellobserver_t_seed{seed}"
    report = OUTPUT_DIR / "reports" / f"dwellobserver_t_seed{seed}_report.md"
    args = [
        sys.executable,
        "scripts/run_dwellobserver_experiments.py",
        "--datasets", "appliance_8ch",
        "--target-modes", "dwell_mean_power",
        "--models", "observer_mlp_prior_only",
        "--window-cycles", "8",
        "--loss-variants", "observer_supervised",
        "--transition-weights", "0",
        "--gain-lambdas", "0",
        "--use-ema-target",
        "--ema-taus", "0.005",
        "--teacher-lambdas", "0.001",
        "--teacher-mask", "unobserved_only",
        "--teacher-space", "residual_delta",
        "--teacher-warmup-epochs", "10",
        "--teacher-ramp-epochs", "20",
        "--teacher-perturbation", "none",
        "--eval-student-teacher-average",
        "--normalization-modes", "branchwise",
        "--output-modes", "residual_dwell",
        "--seed", str(seed),
        "--processed-dir", str(OUTPUT_DIR / "processed"),
        "--output-dir", str(out),
        "--figures-dir", str(fig),
        "--report-path", str(report),
    ]
    if RUN_MODE == "paper":
        args += ["--epochs", "120", "--patience", "15", "--max-train-windows", "5000", "--max-val-windows", "1000", "--hidden-dim", "64", "--num-layers", "2"]
    else:
        args += ["--quick", "--epochs", "2", "--patience", "1", "--max-train-windows", "128", "--max-val-windows", "64"]
    run_command(args)
print("DwellObserver-T rerun completed for seeds:", seeds)

$ python scripts/run_dwellobserver_experiments.py --datasets appliance_8ch --target-modes dwell_mean_power --models observer_mlp_prior_only --window-cycles 8 --loss-variants observer_supervised --transition-weights 0 --gain-lambdas 0 --use-ema-target --ema-taus 0.005 --teacher-lambdas 0.001 --teacher-mask unobserved_only --teacher-space residual_delta --teacher-warmup-epochs 10 --teacher-ramp-epochs 20 --teacher-perturbation none --eval-student-teacher-average --normalization-modes branchwise --output-modes residual_dwell --seed 42 --processed-dir runs/processed --output-dir runs/reconstruction/dwellobserver_t_seed42 --figures-dir runs/figures/dwellobserver_t_seed42 --report-path runs/reports/dwellobserver_t_seed42_report.md --quick --epochs 2 --patience 1 --max-train-windows 128 --max-val-windows 64
<repo root>\scripts\run_dwellobserver_experiments.py:334: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.